# Exercise 2: the same pipeline on a law that has changed

**On your own, at home. One page, and this notebook. Due before session 4.**

Your `rag.py` answered customers from a help centre written for it. Now it
reads the **EU AI Act**: 144 pages of Regulation (EU) 2024/1689, with recitals
before the articles, definitions far from where they are used, and dates
written as cross-references. And a law that has been **amended since**: in
July 2026, Regulation (EU) 2026/1744 changed some of its dates.

Three things go wrong, in turn. You measure each, fix it, and say on one page
what the fix teaches about retrieval on a real document.

**You need** Exercise 1's `rag.py`, working: this notebook imports it. Cells
marked `GIVEN` run as they are. **Cost:** about $0.02. Embedding the Act takes
a minute or two on a laptop.

In [ ]:
# GIVEN: run this cell as it is. It downloads the two regulations once (and checks them),
# loads the model, and defines three helpers you will use: chunk_by_article, bm25_scores, rrf.
%load_ext autoreload
%autoreload 2
import hashlib
import math
import re
import urllib.request
from pathlib import Path

import anthropic
import numpy as np
from dotenv import load_dotenv
from sentence_transformers import SentenceTransformer

import rag

load_dotenv()
client = anthropic.Anthropic()
model = SentenceTransformer("intfloat/multilingual-e5-small")

# From the EU's Publications Office, which serves the Official Journal's PDFs.
LAW = Path("data/law"); LAW.mkdir(parents=True, exist_ok=True)
SOURCES = {
    "aiact.pdf": ("dc8116a1-3fe6-11ef-865a-01aa75ed71a1", "bba630444b3278e881066774002a1d7824308934f49ccfa203e65be43692f55e"),
    "omnibus.pdf": ("b459c07f-86fb-11f1-bf5e-01aa75ed71a1", "0bea4d808256b08275777949ba9cb3c70b7d02e4ebb3ac9b205671b1f552d386"),
}
for name, (cellar, digest) in SOURCES.items():
    path = LAW / name
    if not path.exists():
        url = (f"https://op.europa.eu/o/opportal-service/download-handler?identifier={cellar}"
               "&format=pdf&language=en&productionSystem=cellar")
        req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
        path.write_bytes(urllib.request.urlopen(req).read())
    assert hashlib.sha256(path.read_bytes()).hexdigest() == digest, f"{name}: not the published file"


def chunk_by_article(text_raw, doc):
    """The recitals, then one block per article and per annex, each cut by your rag.chunk when long.
    pypdf puts each heading on a line of its own ("Article 113", "ANNEX III"): split there BEFORE
    joining lines, or a heading looks like any reference to it in a sentence."""
    out, seen = [], set()
    for part in re.split(r"\n(?=(?:Article \d+[a-z]?|ANNEX [IVXL]+)\n)", text_raw):
        head = re.match(r"(Article \d+[a-z]?|ANNEX [IVXL]+)\n", part)
        label = f"{doc} " + (head.group(1).replace("ANNEX", "Annex") if head else "recitals")
        while label in seen:          # the Act prints one heading twice
            label += "'"
        seen.add(label)
        out += rag.chunk(label, re.sub(r"(?<!\n)\n(?!\n)", " ", part), model.tokenizer)
    return out


def bm25_scores(query, texts, k1=1.5, b=0.75):
    """Okapi BM25, by hand: a keyword score of query against every text."""
    words = lambda s: re.findall(r"[a-z0-9]+", s.lower())
    docs = [words(t) for t in texts]
    avg, n, df = sum(map(len, docs)) / len(docs), len(docs), {}
    for d in docs:
        for w in set(d):
            df[w] = df.get(w, 0) + 1
    scores = []
    for d in docs:
        tf = {}
        for w in d:
            tf[w] = tf.get(w, 0) + 1
        scores.append(sum(math.log(1 + (n - df[w] + 0.5) / (df[w] + 0.5)) * tf[w] * (k1 + 1)
                          / (tf[w] + k1 * (1 - b + b * len(d) / avg)) for w in words(query) if w in tf))
    return scores


def rrf(*rankings, k=60):
    """Reciprocal rank fusion: several rankings (lists of positions, best first) into one."""
    score = {}
    for ranking in rankings:
        for r, i in enumerate(ranking):
            score[i] = score.get(i, 0) + 1 / (k + r + 1)
    return sorted(score, key=score.get, reverse=True)


raw = {name: "\n".join(p.extract_text() for p in __import__("pypdf").PdfReader(LAW / name).pages) for name in SOURCES}
print("AI Act:", len(raw["aiact.pdf"]), "characters | amending regulation:", len(raw["omnibus.pdf"]))

## 1. The question nobody can find

Your `rag.py`, unchanged: sentence chunks, the index, the top five. The
question is one any reader of the Act would ask.

In [ ]:
# GIVEN: your pipeline on the 2024 Act. Which passages come back, and what is answered?
LAW_PROMPT = """Answer the question using ONLY the passages below, from EU legal texts. Each passage starts with
its id in square brackets. In "citations", list the ids your answer uses. If the passages do not answer
the question, set "answerable" to false.

{context}"""
DATES_Q = "From when do the obligations for high-risk AI systems listed in Annex III apply?"

plain_chunks = rag.chunk("AI Act", rag.load_pdf(LAW / "aiact.pdf"), model.tokenizer)
plain_index = rag.Index(plain_chunks, model)
passages = plain_index.search(DATES_Q, k=5)
print(len(plain_chunks), "chunks | retrieved:", [p["id"] for p in passages])
ans, _ = rag.answer(client, DATES_Q, passages, system=LAW_PROMPT)
print(f"answerable: {ans.answerable}\n{ans.answer}")

**What happened (one sentence)?** Was the answer found, and if not, was the
model wrong or was the passage missing?

### Ask in the law's words

The dates are in Article 113, "Entry into force and application". Below, the
Act cut one article per chunk, and four ways of searching it. First write the
fourth: a rewrite of the question into the regulation's own vocabulary.

In [ ]:
# Write rewrite(client, question) that asks Claude Haiku 4.5 (model rag.MODEL, max_tokens 100) to turn
# the question into a short keyword query in the vocabulary of an EU regulation (terms such as
# "entry into force", "date of application", "Chapter", "Article"), and returns the query text only.


In [ ]:
# GIVEN: where does Article 113 rank, out of every chunk, with each way of searching?
art_chunks = chunk_by_article(raw["aiact.pdf"], "AI Act")
art_index = rag.Index(art_chunks, model)
is_113 = lambda c: c["id"].startswith("AI Act Article 113")
position = {c["id"]: i for i, c in enumerate(art_chunks)}
rank = lambda ranking: next(r + 1 for r, i in enumerate(ranking) if is_113(art_chunks[i]))
texts = [c["text"] for c in art_chunks]
by_score = lambda scores: sorted(range(len(scores)), key=lambda i: -scores[i])
for q in [DATES_Q, "What is the date of application of Chapter III on high-risk AI systems?"]:
    dense = [position[p["id"]] for p in art_index.search(q, k=len(art_chunks))]
    keyword = by_score(bm25_scores(q, texts))
    rq = rewrite(client, q)
    rewritten = by_score(bm25_scores(rq, texts))
    print(f"{q}\n   rewritten: {rq!r}")
    print(f"   rank of Article 113 of {len(texts)}: meaning {rank(dense)} | keywords {rank(keyword)} | "
          f"both (fused) {rank(rrf(dense, keyword))} | keywords of the rewrite {rank(rewritten)}")

**The structural fix.** In an EU regulation, the dates are always in the last
article, "Entry into force and application". When a question asks *when*,
your code can add that article to the passages, whatever the search returns.

In [ ]:
# Write route(question, passages) that returns the passages, plus every chunk of Article 113
# (is_113 tells you which) at the front, when the question asks about dates: when it contains
# any of "when", "date", "from what", "apply", "applies" (lower case). Otherwise return them as they are.
# Then answer DATES_Q from route(DATES_Q, art_index.search(DATES_Q, k=5)) with LAW_PROMPT, and print the
# answer and its citations.


**Is that answer right (two sentences)?** It cites the right article of the
2024 text. Is it still true in October 2026?

## 2. Recitals are not the law

The Act's first 180 recitals explain its intentions; only the articles are law.
Look at what plain retrieval returns for a definition question.

In [ ]:
# GIVEN: for a definition, how many of the top five are recitals rather than articles?
q = "What is a provider of an AI system?"
got = art_index.search(q, k=5)
print([p["id"] for p in got])
print(f"recitals: {sum('recitals' in p['id'] for p in got)} of 5")

**When would a recital be the wrong source (two sentences)?** And how would you
make your pipeline prefer articles?

## 3. The law has changed: add the amendment

Regulation (EU) 2026/1744 amends Article 113, among others. Add it to the
corpus, and tell the model which text is later.

In [ ]:
# Chunk the amending regulation with chunk_by_article(raw["omnibus.pdf"], "Amending Regulation 2026/1744"),
# build one rag.Index over the AI Act's article chunks plus these, retrieve the top five for DATES_Q,
# route() them, and answer with a prompt that adds, before the passages:
#   "The AI Act is Regulation (EU) 2024/1689 of 13 June 2024. Regulation (EU) 2026/1744 of 8 July 2026 amends it:
#    where the two disagree, the amending regulation applies. Say which text your answer comes from."
# Print the answer and its citations.


**Your page (hand it in as a PDF).** Three paragraphs, one per failure: what
went wrong, what you measured, and what the fix teaches about retrieval on a
real document. Then one sentence: what would you put in place so that a
pipeline in production knows when its documents have been superseded?